# Example Literature Project

## Helper Functions

In [ ]:
import json
import textwrap
from pathlib import Path
from time import perf_counter

from kilin.conversations import ConversationStore
from kilin.rag import RAGSession


def _print_wrapped(label, value, width=79):
    """Print text with a fixed display width, preserving paragraphs."""
    print(f"{label}:")
    for line in str(value).splitlines() or [""]:
        if not line.strip():
            print()
        else:
            print(textwrap.fill(
                line, width=width, break_long_words=False,
                break_on_hyphens=False,
            ))
    print("")


def create_conversation(project_dir):
    """Create an independent conversation and its active RAG session.

    Parameters
    ----------
    project_dir : str or Path
        Existing KíLín project directory.

    Returns
    -------
    tuple
        Conversation store, conversation record, and active session.
        Close the session after the last question.

    """
    store = ConversationStore(Path(project_dir))
    conversation = store.create()
    path = store.state_path(conversation)
    session = RAGSession(Path(project_dir), conversation_path=path)
    print(f"Conversation: {path}")
    return store, conversation, session


def ask_question(session, question):
    """Ask the active model and print a width-limited exchange.

    Parameters
    ----------
    session : RAGSession
        Active conversation session.
    question : str
        User question.

    Returns
    -------
    str
        Assistant response.

    """
    print("\n" + "=" * 79)
    _print_wrapped("USER", question)
    start = perf_counter()
    try:
        answer = session.ask(question)
    except Exception:
        print(f"Elapsed before failure: {perf_counter() - start:.3f} s")
        raise
    _print_wrapped("ASSISTANT", answer)
    print(f"Wall time: {perf_counter() - start:.3f} s")
    return answer


def _print_retrieved_chunks(stages, width=79):
    """Display retrieved passages recorded in the native trace.

    Parameters
    ----------
    stages : dict
        Stages of one recorded conversation turn.
    width : int, optional
        Maximum display width.

    """
    retrieval = stages.get("03_retrieval", {})
    operations = retrieval.get("operations", {})

    print("\nRETRIEVED CHUNKS")

    found = 0

    def visit(value, path):
        nonlocal found

        if isinstance(value, dict):
            text = next(
                (
                    value[key]
                    for key in (
                        "text", "content", "chunk_text",
                        "passage", "snippet",
                    )
                    if isinstance(value.get(key), str)
                    and value[key].strip()
                ),
                None,
            )

            if text is not None:
                found += 1
                print(f"\n  CHUNK {found}")
                print(f"  Trace location: {path}")

                for key, item in value.items():
                    if key in (
                        "text", "content", "chunk_text",
                        "passage", "snippet",
                    ):
                        continue

                    if isinstance(item, (str, int, float, bool)):
                        _print_wrapped(f"  {key}", item, width)

                _print_wrapped("  PASSAGE", text, width)
                return

            for key, child in value.items():
                visit(child, f"{path}/{key}")

        elif isinstance(value, list):
            for index, child in enumerate(value):
                visit(child, f"{path}/{index}")

    for name, operation in operations.items():
        if "retrieval" not in name.lower():
            continue

        visit(operation.get("details", {}), name)

    if not found:
        print("  No passage text found in the retrieval trace.")


def inspect_conversation(
    conversation_path,
    turn_number=None,
    show_chunks=True,
):
    """Display routing, retrieval, model calls, tools, and trace timings.

    Parameters
    ----------
    conversation_path : str or Path
        Native KíLín trace JSON file.
    turn_number : int, optional
        One-based turn index, or None for all turns.
    show_chunks : bool, optional
        Whether to display passages recorded in the retrieval trace.
        Defaults to True.

    Returns
    -------
    dict
        Unmodified native trace record.

    """
    with Path(conversation_path).open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = list(trace["sessions"]["05_conversations"]["turns"].items())
    if turn_number is not None:
        if not 1 <= turn_number <= len(turns):
            raise ValueError("Invalid turn number.")
        turns = [turns[turn_number - 1]]

    def operation(stages, stage_name, operation_name):
        return (stages.get(stage_name, {})
                .get("operations", {}).get(operation_name, {}))

    def show_fields(heading, data, keys):
        print(f"\n{heading}")
        for key in keys:
            if key in data:
                _print_wrapped(f"  {key}", data[key])

    for turn_id, turn in turns:
        stages = turn.get("stages", {})
        print("\n" + "=" * 79)
        print(f"TURN: {turn_id}")
        print("=" * 79)

        question = operation(stages, "01_question", "01_turn")
        response = operation(
            stages, "01_question", "09_conversation_turn_representation"
        )
        show_fields("EXCHANGE", question.get("details", {}), ["question"])
        show_fields("ANSWER", response.get("details", {}), ["summary"])

        decision = operation(
            stages, "01_question", "05_intent_classification_decided"
        ).get("details", {})
        show_fields("INTENT", decision.get("classification", {}), [
            "intent", "standalone", "refers_back", "related_turn_ids"
        ])
        show_fields("ROUTING", decision, ["shortcut", "fallback"])
        show_fields("RESOLVED POLICY", decision.get("resolved", {}), [
            "history_policy", "retrieval_mode", "retrieval_strategy"
        ])
        settings = operation(
            stages, "01_question", "02_turn_settings_snapshot"
        ).get("details", {})
        show_fields("EFFECTIVE SETTINGS", settings, [
            "project_type_id", "backend", "chat_model", "router_model",
            "thinking", "temperature", "max_new_tokens", "retrieval_count"
        ])
        retrieval = operation(
            stages, "03_retrieval", "02_retrieval_result"
        )
        show_fields("RETRIEVAL", retrieval.get("details", {}), [
            "mode", "strategy", "evidence_count", "duration_ms"
        ])
        if show_chunks:
            _print_retrieved_chunks(stages)
            
        calls = []
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                if name == "01_llm_call_input" or name == "02_llm_call_output":
                    calls.append((stage_name, name, entry))
                elif "tool" in name.lower():
                    print(f"\nTOOL OPERATION: {stage_name}/{name}")
                    show_fields("  DETAILS", entry.get("details", {}), [
                        "tool", "tool_name", "status", "duration_ms"
                    ])
        for stage_name, name, entry in calls:
            details = entry.get("details", {})
            show_fields(f"MODEL CALL: {stage_name}/{name}", details, [
                "call_id", "model_id", "backend", "device", "thinking",
                "input_token_count", "prompt_token_count",
                "output_token_count", "finish_reason", "reply_duration_ms"
            ])
            show_fields("GENERATION SETTINGS", details.get("generation", {}), [
                "temperature", "max_new_tokens", "thinking"
            ])

        print("\nOPERATION TIMINGS (ms)")
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                timing = entry.get("timing", {})
                duration = timing.get("operation_duration_ms")
                if duration is not None:
                    print(f"  {stage_name}/{name}: {duration:.3f}")

    return trace


## Create Empty Literature Project

In [ ]:
from pathlib import Path

from kilin.workspace import prepare_sources_workspace

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run this notebook from within the KíLín repository.")
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
)

print(f"Created literature project: {project.root}")
print(f"Source directory: {project.source}")

## First Conversation without Sources

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(session, 'What is Brownian motion?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(session, 'Which articles are available in this project?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(session, 'Summarize the articles in this project.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(session, 'Find articles about neural networks in this project.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
session.close()

## Add Articles to Project

In [ ]:
from kilin.workspace import add_project_sources

fixtures = repo_root / "tests" / "fixtures"

articles = sorted(fixtures.glob("*.pdf"))

if not articles:
    raise FileNotFoundError(f"No PDF articles found in {fixtures}")

project = add_project_sources(
    project_dir,
    articles,
)

print(f"Project: {project.root}")
print(f"Imported articles: {len(articles)}")

for article in sorted(project.source.glob("*.pdf")):
    print(f"  {article.name}")

## Document Discovery and Retrieval

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, 'Which articles are available in this project?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, 'Which of these articles discuss BRAPH?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(
    session,
    'Show me the source information for the BRAPH 2 article.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(
    session,
    'Find passages discussing graph measures in the BRAPH articles.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


In [ ]:
answer = ask_question(
    session,
    'What do those passages say about their implementation?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(session, 'Which articles discuss deep learning?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
session.close()

## Article Understanding and Synthesis

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, 'Give me an overview of the BRAPH 2 paper.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


In [ ]:
answer = ask_question(
    session,
    'Summarize its main contributions and limitations.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(
    session,
    'Generate a detailed summary of the BRAPH 2 paper.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(
    session,
    'How does BRAPH 2 differ from the earlier BRAPH framework?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(session, 'What evidence supports those differences?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(
    session,
    'Which of the four papers is most relevant to graph-based analysis '
    'of brain connectivity, and why?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
session.close()

## Specialized Literature Tools

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(
    session,
    'Find the passage in the BRAPH 2 paper '
    'describing its software architecture.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(
    session,
    'What programming language is BRAPH 2 implemented in?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(session, 'List the articles that mention graph theory.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(
    session,
    'Filter those results to articles discussing brain connectivity.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(
    session,
    'Classify the five articles by their main research topic.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(
    session,
    'Review the five articles for their relevance to a literature review '
    'on computational brain connectivity.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
answer = ask_question(
    session,
    'Check whether the BRAPH 2 paper supports the claim '
    'that it provides a graphical user interface.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)

In [ ]:
answer = ask_question(
    session,
    'Which article should I cite for the introduction of the BRAPH framework?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)

In [ ]:
session.close()

## Continuity and Robustness

In [ ]:
store, conversation, session = create_conversation(project_dir)

In [ ]:
answer = ask_question(session, 'Summarize the BRAPH 2 article.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, 'What are its main limitations?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
answer = ask_question(session, 'How does it compare with the 2017 paper?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)

In [ ]:
answer = ask_question(session, 'Now tell me about the deep learning article.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)

In [ ]:
answer = ask_question(session, 'What methods does it use?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)

In [ ]:
answer = ask_question(session, 'Compare it with the previous article.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)

In [ ]:
answer = ask_question(
    session,
    'Find evidence that all five articles use the same dataset.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)

In [ ]:
answer = ask_question(session, 'Which article did we discuss first?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)

In [ ]:
session.close()